In [1]:
import pandas as pd
import numpy as np
import plotly as py

In [2]:
df= pd.read_csv('nepse.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.sort_values(by='Date').reset_index(drop=True,inplace=True)


In [3]:
df

,Symbol,Date,Open,High,Low,Close,Percent Change,Volume,Turn Over
0,NEPSE,2026-03-30,2881.98,2891.53,2821.82,2831.39,-1.65 %,"12,966,659,168.04",-
1,NEPSE,2026-03-29,2958.78,2958.79,2872.99,2879.11,-2.40 %,"15,033,310,867.98",-
2,NEPSE,2026-03-26,2933.89,2951.18,2918.36,2950.50,0.49 %,"13,148,511,094.97",-
3,NEPSE,2026-03-25,2964.68,2969.50,2929.44,2935.94,-0.82 %,"15,090,416,846.15",-
4,NEPSE,2026-03-24,2943.47,2960.40,2928.34,2960.40,0.81 %,"14,839,183,643.29",-
...,...,...,...,...,...,...,...,...,...
1152,NEPSE,2021-04-06,2678.96,2678.96,2641.80,2674.47,0.00 %,"7,477,795,737.92",-
1153,NEPSE,2021-04-05,2679.39,2682.27,2639.67,2658.75,0.00 %,"7,296,760,155.82",-
1154,NEPSE,2021-04-04,2658.31,2658.73,2646.05,2657.61,0.00 %,"9,006,858,557.96",-
1155,NEPSE,2021-04-01,2645.50,2645.50,2615.26,2631.90,0.00 %,"7,250,924,123.60",-


In [4]:
df= df[['Symbol','Date', 'Open', 'High', 'Low', 'Close', 'Volume']]

In [5]:
df

,Symbol,Date,Open,High,Low,Close,Volume
0,NEPSE,2026-03-30,2881.98,2891.53,2821.82,2831.39,"12,966,659,168.04"
1,NEPSE,2026-03-29,2958.78,2958.79,2872.99,2879.11,"15,033,310,867.98"
2,NEPSE,2026-03-26,2933.89,2951.18,2918.36,2950.50,"13,148,511,094.97"
3,NEPSE,2026-03-25,2964.68,2969.50,2929.44,2935.94,"15,090,416,846.15"
4,NEPSE,2026-03-24,2943.47,2960.40,2928.34,2960.40,"14,839,183,643.29"
...,...,...,...,...,...,...,...
1152,NEPSE,2021-04-06,2678.96,2678.96,2641.80,2674.47,"7,477,795,737.92"
1153,NEPSE,2021-04-05,2679.39,2682.27,2639.67,2658.75,"7,296,760,155.82"
1154,NEPSE,2021-04-04,2658.31,2658.73,2646.05,2657.61,"9,006,858,557.96"
1155,NEPSE,2021-04-01,2645.50,2645.50,2615.26,2631.90,"7,250,924,123.60"


In [6]:
import plotly.graph_objects as go

# Optional: filter one symbol; comment this out to plot all rows in df
symbol = df['Symbol'].iloc[0]
plot_df = df[df['Symbol'] == symbol].copy()

fig = go.Figure(
    data=[
        go.Candlestick(
            x=plot_df['Date'],
            open=plot_df['Open'],
            high=plot_df['High'],
            low=plot_df['Low'],
            close=plot_df['Close'],
            name=symbol,
            increasing_line_color='green',
            decreasing_line_color='red'
        )
    ]
)

fig.update_layout(
    title=f"{symbol} Candlestick Chart",
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False,
    template='plotly_white',
    height=600
)

fig.show()

In [7]:
# Fair Value Gap (FVG) detection using a 3-candle structure
# Bullish FVG at i: Low[i] > High[i-2]
# Bearish FVG at i: High[i] < Low[i-2]
# Keep only gaps where size >= 0.5% of the reference price

MIN_GAP_PCT = 0.5

fvg_df = df.copy()
fvg_df = fvg_df.sort_values(['Symbol', 'Date']).reset_index(drop=True)

records = []

for sym, g in fvg_df.groupby('Symbol', sort=False):
    g = g.reset_index(drop=True)

    for i in range(2, len(g)):
        c0 = g.loc[i - 2]
        c2 = g.loc[i]

        # Bullish imbalance gap: price jumped up, leaving a gap below current candle
        if c2['Low'] > c0['High']:
            gap_size = c2['Low'] - c0['High']
            gap_pct = (gap_size / c0['High']) * 100

            if gap_pct >= MIN_GAP_PCT:
                records.append({
                    'Symbol': sym,
                    'Date': c2['Date'],
                    'Type': 'Bullish',
                    'Gap_Low': c0['High'],
                    'Gap_High': c2['Low'],
                    'Gap_Size': gap_size,
                    'Gap_Pct': gap_pct,
                    'Bar_Index': i
                })

        # Bearish imbalance gap: price dropped down, leaving a gap above current candle
        if c2['High'] < c0['Low']:
            gap_size = c0['Low'] - c2['High']
            gap_pct = (gap_size / c0['Low']) * 100

            if gap_pct >= MIN_GAP_PCT:
                records.append({
                    'Symbol': sym,
                    'Date': c2['Date'],
                    'Type': 'Bearish',
                    'Gap_Low': c2['High'],
                    'Gap_High': c0['Low'],
                    'Gap_Size': gap_size,
                    'Gap_Pct': gap_pct,
                    'Bar_Index': i
                })

fvg = pd.DataFrame(records)

# For continuous FVGs (consecutive bars), keep only the one with highest Gap_Size
if not fvg.empty:
    fvg = fvg.sort_values(['Symbol', 'Type', 'Bar_Index']).reset_index(drop=True)

    fvg['Is_Continuous'] = (
        (fvg['Symbol'] == fvg['Symbol'].shift(1)) &
        (fvg['Type'] == fvg['Type'].shift(1)) &
        ((fvg['Bar_Index'] - fvg['Bar_Index'].shift(1)) == 1)
    )

    fvg['Continuous_Group'] = (~fvg['Is_Continuous']).cumsum()

    fvg = (
        fvg.loc[fvg.groupby(['Symbol', 'Type', 'Continuous_Group'])['Gap_Size'].idxmax()]
        .sort_values(['Symbol', 'Date'])
        .reset_index(drop=True)
    )

print(f"Total FVGs found (>= {MIN_GAP_PCT}% and deduped continuous): {len(fvg)}")
fvg.head(20)

Total FVGs found (>= 0.5% and deduped continuous): 189


,Symbol,Date,Type,Gap_Low,Gap_High,Gap_Size,Gap_Pct,Bar_Index,Is_Continuous,Continuous_Group
0,NEPSE,2021-04-04,Bullish,2619.15,2646.05,26.90,1.027051,2,False,92
1,NEPSE,2021-04-15,Bullish,2694.38,2731.82,37.44,1.389559,9,False,93
2,NEPSE,2021-04-19,Bearish,2716.87,2731.82,14.95,0.547254,11,False,1
3,NEPSE,2021-04-26,Bearish,2579.41,2609.02,29.61,1.134909,16,False,2
4,NEPSE,2021-05-03,Bullish,2612.29,2645.71,33.42,1.279337,21,False,94
5,NEPSE,2021-05-12,Bullish,2647.07,2664.04,16.97,0.641086,28,False,95
6,NEPSE,2021-05-17,Bullish,2702.74,2747.82,45.08,1.667937,31,False,96
7,NEPSE,2021-06-01,Bearish,2799.85,2814.72,14.87,0.528294,41,False,3
8,NEPSE,2021-06-06,Bullish,2827.29,2880.47,53.18,1.880953,44,True,97
9,NEPSE,2021-06-14,Bullish,2971.89,3009.15,37.26,1.253748,50,False,98


In [8]:
import plotly.graph_objects as go

# Use the same symbol selection as before
symbol = df['Symbol'].iloc[0]
plot_df = df[df['Symbol'] == symbol].sort_values('Date').copy()

# Build/refresh candlestick figure
fig = go.Figure(
    data=[
        go.Candlestick(
            x=plot_df['Date'],
            open=plot_df['Open'],
            high=plot_df['High'],
            low=plot_df['Low'],
            close=plot_df['Close'],
            name=symbol,
            increasing_line_color='green',
            decreasing_line_color='red'
        )
    ]
)

# Filter FVGs for the plotted symbol
symbol_fvg = fvg[fvg['Symbol'] == symbol].copy() if 'fvg' in globals() else pd.DataFrame()

if not symbol_fvg.empty:
    bullish = symbol_fvg[symbol_fvg['Type'] == 'Bullish']
    bearish = symbol_fvg[symbol_fvg['Type'] == 'Bearish']

    # Small upward pointer for bullish gaps
    if not bullish.empty:
        fig.add_trace(
            go.Scatter(
                x=bullish['Date'],
                y=bullish['Gap_High'],
                mode='markers',
                name='Bullish FVG',
                marker=dict(
                    symbol='triangle-up',
                    size=8,
                    color='rgba(34, 139, 34, 0.9)',
                    line=dict(color='white', width=0.8)
                ),
                customdata=np.stack([bullish['Gap_Low'], bullish['Gap_High'], bullish['Gap_Size']], axis=-1),
                hovertemplate='Date: %{x}<br>Type: Bullish FVG<br>Gap Low: %{customdata[0]:.2f}<br>Gap High: %{customdata[1]:.2f}<br>Gap Size: %{customdata[2]:.2f}<extra></extra>'
            )
        )

    # Small downward pointer for bearish gaps
    if not bearish.empty:
        fig.add_trace(
            go.Scatter(
                x=bearish['Date'],
                y=bearish['Gap_Low'],
                mode='markers',
                name='Bearish FVG',
                marker=dict(
                    symbol='triangle-down',
                    size=8,
                    color='rgba(220, 20, 60, 0.9)',
                    line=dict(color='white', width=0.8)
                ),
                customdata=np.stack([bearish['Gap_Low'], bearish['Gap_High'], bearish['Gap_Size']], axis=-1),
                hovertemplate='Date: %{x}<br>Type: Bearish FVG<br>Gap Low: %{customdata[0]:.2f}<br>Gap High: %{customdata[1]:.2f}<br>Gap Size: %{customdata[2]:.2f}<extra></extra>'
            )
        )

fig.update_layout(
    title=f"{symbol} Candlestick with FVG Pointers",
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False,
    template='plotly_white',
    height=700
)

fig.show()

In [9]:
# FVG performance analysis setup
horizons = [3, 5, 10, 15, 20]

if 'fvg' not in globals() or fvg.empty:
    raise ValueError("`fvg` is empty. Run the FVG detection cell first.")

# Detect a sector column if available
sector_col = next((c for c in df.columns if 'sector' in c.lower()), None)

signal_records = []
performance_records = []

for signal_id, sig in fvg.reset_index(drop=True).iterrows():
    sym = sig['Symbol']
    fvg_type = sig['Type']
    gap_low = float(sig['Gap_Low'])
    gap_high = float(sig['Gap_High'])
    sig_idx = int(sig['Bar_Index'])

    sym_df = df[df['Symbol'] == sym].sort_values('Date').reset_index(drop=True)

    # Need at least one candle after signal candle to search for re-entry
    if sig_idx + 1 >= len(sym_df):
        continue

    after_signal = sym_df.iloc[sig_idx + 1:].copy()

    # Re-entry means any overlap between candle range [Low, High] and FVG zone [gap_low, gap_high]
    touches_gap = (after_signal['Low'] <= gap_high) & (after_signal['High'] >= gap_low)

    if not touches_gap.any():
        continue

    first_touch_offset = np.where(touches_gap.to_numpy())[0][0]
    entry_idx = sig_idx + 1 + first_touch_offset
    entry_row = sym_df.loc[entry_idx]

    entry_date = entry_row['Date']
    entry_price = float(entry_row['Close'])
    sector_value = entry_row[sector_col] if sector_col is not None else 'N/A'

    signal_records.append({
        'Signal_ID': signal_id,
        'Symbol': sym,
        'Sector': sector_value,
        'FVG_Type': fvg_type,
        'Signal_Date': sig['Date'],
        'Entry_Date': entry_date,
        'Entry_Index': entry_idx,
        'Entry_Price': entry_price,
        'Gap_Low': gap_low,
        'Gap_High': gap_high,
        'Gap_Size': float(sig['Gap_Size']),
        'Gap_Pct': float(sig.get('Gap_Pct', np.nan))
    })

    for h in horizons:
        future_idx = entry_idx + h
        if future_idx >= len(sym_df):
            continue

        future_close = float(sym_df.loc[future_idx, 'Close'])
        path = sym_df.iloc[entry_idx + 1:future_idx + 1]

        if fvg_type == 'Bullish':
            ret_pct = ((future_close - entry_price) / entry_price) * 100
            mfe_pct = ((path['High'].max() - entry_price) / entry_price) * 100
            mae_pct = ((path['Low'].min() - entry_price) / entry_price) * 100
        else:
            ret_pct = ((entry_price - future_close) / entry_price) * 100
            mfe_pct = ((entry_price - path['Low'].min()) / entry_price) * 100
            mae_pct = ((entry_price - path['High'].max()) / entry_price) * 100

        rr = np.nan if mae_pct >= 0 else (mfe_pct / abs(mae_pct))

        performance_records.append({
            'Signal_ID': signal_id,
            'Symbol': sym,
            'Sector': sector_value,
            'FVG_Type': fvg_type,
            'Horizon': h,
            'Return_Pct': ret_pct,
            'MFE_Pct': mfe_pct,
            'MAE_Pct': mae_pct,
            'RR': rr,
            'Win': int(ret_pct > 0)
        })

entry_points = pd.DataFrame(signal_records)
perf = pd.DataFrame(performance_records)

print(f"Total detected FVG signals: {len(fvg)}")
print(f"Signals with a valid re-entry entry point: {entry_points['Signal_ID'].nunique() if not entry_points.empty else 0}")
print(f"Signals contributing to at least one horizon metric: {perf['Signal_ID'].nunique() if not perf.empty else 0}")

Total detected FVG signals: 189
Signals with a valid re-entry entry point: 182
Signals contributing to at least one horizon metric: 181


In [ ]:
if perf.empty:
    print("No valid FVG entries with enough forward candles for the selected horizons.")
else:
    summary_by_type = (
        perf.groupby(['FVG_Type', 'Horizon'], as_index=False)
        .agg(
            Signals=('Return_Pct', 'count'),
            Avg_Return_Pct=('Return_Pct', lambda x: x.mean() * 100),
            Win_Rate_Pct=('Win', lambda x: x.mean() * 100),
            Avg_MFE_Pct=('MFE_Pct', 'mean'),
            Avg_MAE_Pct=('MAE_Pct', 'mean'),
            Avg_RR=('RR', 'mean'),
            Median_RR=('RR', 'median')
        )
        .sort_values(['FVG_Type', 'Horizon'])
        .reset_index(drop=True)
    )

    overall_by_horizon = (
        perf.groupby('Horizon', as_index=False)
        .agg(
            Signals=('Return_Pct', 'count'),
            Avg_Return_Pct=('Return_Pct', lambda x: x.mean() * 100),
            Win_Rate_Pct=('Win', lambda x: x.mean() * 100),
            Avg_MFE_Pct=('MFE_Pct', 'mean'),
            Avg_MAE_Pct=('MAE_Pct', 'mean'),
            Avg_RR=('RR', 'mean')
        )
        .sort_values('Horizon')
        .reset_index(drop=True)
    )

    print("Overall performance by horizon")
    display(overall_by_horizon)

    print("Performance split by Bullish vs Bearish FVG")
    display(summary_by_type)

    if sector_col is not None:
        summary_by_sector = (
            perf.groupby(['Sector', 'FVG_Type', 'Horizon'], as_index=False)
            .agg(
                Signals=('Return_Pct', 'count'),
                Avg_Return_Pct=('Return_Pct', lambda x: x.mean() * 100),
                Win_Rate_Pct=('Win', lambda x: x.mean() * 100),
                Avg_MFE_Pct=('MFE_Pct', 'mean'),
                Avg_MAE_Pct=('MAE_Pct', 'mean'),
                Avg_RR=('RR', 'mean')
            )
            .sort_values(['Sector', 'FVG_Type', 'Horizon'])
            .reset_index(drop=True)
        )

        print(f"Performance by sector (column detected: {sector_col})")
        display(summary_by_sector)
    else:
        print("Sector split skipped: no sector column found in dataset.")

Overall performance by horizon


,Horizon,Signals,Avg_Return_Pct,Win_Rate_Pct,Avg_MFE_Pct,Avg_MAE_Pct,Avg_RR
0,3,181,0.212120,51.933702,1.999708,-1.664393,25.365736
1,5,181,-0.027836,47.513812,2.574938,-2.381209,27.618142
2,10,181,0.023694,45.856354,3.704052,-3.551667,29.002697
3,15,180,-0.319442,45.000000,4.529232,-4.609458,10.096288
4,20,179,-0.546159,46.927374,5.250327,-5.353851,9.846065


Performance split by Bullish vs Bearish FVG


,FVG_Type,Horizon,Signals,Avg_Return_Pct,Win_Rate_Pct,Avg_MFE_Pct,Avg_MAE_Pct,Avg_RR,Median_RR
0,Bearish,3,89,0.523930,57.303371,2.058926,-1.664172,8.340302,1.245972
1,Bearish,5,89,0.049163,53.932584,2.653417,-2.445941,8.065958,1.577264
2,Bearish,10,89,0.183388,51.685393,3.864890,-3.845151,9.144208,1.265514
3,Bearish,15,89,-0.049889,47.191011,4.626293,-4.845497,10.842740,0.977706
4,Bearish,20,88,-0.301724,50.000000,5.352148,-5.526875,11.942338,0.916563
5,Bullish,3,92,-0.089522,46.739130,1.942421,-1.664607,42.992067,0.651337
6,Bullish,5,92,-0.102325,41.304348,2.499019,-2.318587,46.950639,0.755039
7,Bullish,10,92,-0.130793,40.217391,3.548459,-3.267752,48.206510,1.006354
8,Bullish,15,91,-0.583071,42.857143,4.434303,-4.378606,9.366423,0.908282
9,Bullish,20,91,-0.782536,43.956044,5.151862,-5.186530,7.819668,0.908485


Sector split skipped: no sector column found in dataset.


In [12]:
if entry_points.empty:
    print("No valid entry points were found.")
else:
    print("All entry points")
    display(entry_points.sort_values(['Symbol', 'Entry_Date']).reset_index(drop=True))

All entry points


,Signal_ID,Symbol,Sector,FVG_Type,Signal_Date,Entry_Date,Entry_Index,Entry_Price,Gap_Low,Gap_High,Gap_Size,Gap_Pct
0,0,NEPSE,N/A,Bullish,2021-04-04,2021-04-05,3,2658.75,2619.15,2646.05,26.90,1.027051
1,1,NEPSE,N/A,Bullish,2021-04-15,2021-04-18,10,2699.51,2694.38,2731.82,37.44,1.389559
2,3,NEPSE,N/A,Bearish,2021-04-26,2021-04-27,17,2599.08,2579.41,2609.02,29.61,1.134909
3,4,NEPSE,N/A,Bullish,2021-05-03,2021-05-04,22,2654.03,2612.29,2645.71,33.42,1.279337
4,2,NEPSE,N/A,Bearish,2021-04-19,2021-05-16,30,2738.30,2716.87,2731.82,14.95,0.547254
...,...,...,...,...,...,...,...,...,...,...,...,...
177,184,NEPSE,N/A,Bullish,2026-01-26,2026-01-27,1120,2726.51,2714.67,2746.87,32.20,1.186148
178,183,NEPSE,N/A,Bullish,2026-01-20,2026-02-03,1125,2683.15,2648.98,2678.81,29.83,1.126094
179,185,NEPSE,N/A,Bearish,2026-02-22,2026-02-25,1138,2648.90,2642.37,2657.67,15.30,0.575692
180,186,NEPSE,N/A,Bullish,2026-03-10,2026-03-11,1144,2796.57,2712.60,2796.13,83.53,3.079333
